# Phase 2 — Unified Neural Network Training (RNN + LSTM + BiLSTM)

This notebook trains **all three models sequentially** on the full Samanantar en-hi dataset.
At the end, a **comparison table** shows BLEU, ChrF++, BERTScore and COMET for all models.

### Models:
- **RNN** + Self-Attention (Encoder + Decoder)
- **LSTM** + Self-Attention (Encoder + Decoder)
- **BiLSTM** + Cross-Attention (Encoder + Decoder with LayerNorm + residual)

### Pipeline:
1. Shared imports, config, data loading
2. Shared Dataset / DataLoader
3. Model definitions (RNN, LSTM, BiLSTM)
4. Shared train/eval loop
5. Train all 3 models one by one
6. Evaluate all 3 on test set
7. Final comparison table

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import os
import sys
import gc
import math
import glob
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm.auto import tqdm

# Fix for Jupyter not picking up mt_env packages
sys.path.append(r'C:\Users\Dr-Saurabh\mt\Lib\site-packages')

print('All imports successful!')

In [ ]:
# ── Cell 2: Shared Config ─────────────────────────────────────────────────────
DATA_DIR  = './phase2_data/'
SAVE_DIR  = './phase2_checkpoints/'
os.makedirs(SAVE_DIR, exist_ok=True)

# Shared hyperparameters
BATCH_SIZE    = 256
EMBED_DIM     = 300
HIDDEN_DIM    = 512   # BiLSTM uses 256 (outputs 512 bidirectionally)
DROPOUT       = 0.3
LEARNING_RATE = 0.0005
N_EPOCHS      = 10
CLIP          = 1.0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
# ── Cell 3: Load Vocabularies + Embeddings ────────────────────────────────────
with open(os.path.join(DATA_DIR, 'eng_vocab.pkl'), 'rb') as f:
    eng_vocab = pickle.load(f)
with open(os.path.join(DATA_DIR, 'hi_vocab.pkl'), 'rb') as f:
    hi_vocab = pickle.load(f)
with open(os.path.join(DATA_DIR, 'preprocessing_config.pkl'), 'rb') as f:
    pp_config = pickle.load(f)

MAX_LEN    = pp_config['MAX_LEN']
eng_matrix = np.load(os.path.join(DATA_DIR, 'eng_embedding_matrix.npy'))
hi_matrix  = np.load(os.path.join(DATA_DIR, 'hi_embedding_matrix.npy'))

# Reverse vocab for decoding
idx2hi  = {idx: word for word, idx in hi_vocab.items()}
idx2eng = {idx: word for word, idx in eng_vocab.items()}

print(f'English vocab : {len(eng_vocab):,}  | Matrix: {eng_matrix.shape}')
print(f'Hindi vocab   : {len(hi_vocab):,}  | Matrix: {hi_matrix.shape}')
print(f'MAX_LEN       : {MAX_LEN}')

In [ ]:
# ── Cell 4: Dataset + DataLoaders (shared by all models) ─────────────────────
class ChunkedTranslationDataset(Dataset):
    """Loads one pre-processed CSV chunk into memory as long tensors."""
    def __init__(self, csv_path, max_len):
        df = pd.read_csv(csv_path)
        self.english = torch.tensor(
            [list(map(int, row.split())) for row in df['eng_padded']],
            dtype=torch.long
        )
        self.hindi = torch.tensor(
            [list(map(int, row.split())) for row in df['hi_padded']],
            dtype=torch.long
        )
        assert self.english.shape[1] == max_len, \
            f'Expected MAX_LEN={max_len} but got {self.english.shape[1]} in {csv_path}'

    def __len__(self):
        return len(self.english)

    def __getitem__(self, idx):
        return self.english[idx], self.hindi[idx]


def make_dataloader(split_name, shuffle):
    """Build a DataLoader over all processed chunks for a given split."""
    chunk_files = sorted(glob.glob(
        os.path.join(DATA_DIR, f'{split_name}_processed_chunk*.csv')
    ))
    assert chunk_files, f"No processed chunks found for split '{split_name}' in {DATA_DIR}"
    datasets = [ChunkedTranslationDataset(f, MAX_LEN) for f in chunk_files]
    combined = ConcatDataset(datasets)
    loader = DataLoader(
        combined,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        pin_memory=(device.type == 'cuda'),
        num_workers=4
    )
    print(f'  {split_name}: {len(combined):,} pairs | {len(loader):,} batches')
    return loader

print('Building DataLoaders...')
train_loader = make_dataloader('train', shuffle=True)
val_loader   = make_dataloader('val',   shuffle=False)
test_loader  = make_dataloader('test',  shuffle=False)
print('Done!')

In [ ]:
# ── Cell 5: RNN Model Definition ─────────────────────────────────────────────

class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, pretrained_embeddings):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(torch.tensor(pretrained_embeddings, dtype=torch.float32))
        self.embedding.weight.requires_grad = True
        self.rnn = nn.RNN(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)
        self.attention_q = nn.Linear(hidden_dim, hidden_dim)
        self.attention_k = nn.Linear(hidden_dim, hidden_dim)
        self.attention_v = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale   = math.sqrt(hidden_dim)

    def self_attention(self, hidden_states):
        Q = self.attention_q(hidden_states)
        K = self.attention_k(hidden_states)
        V = self.attention_v(hidden_states)
        scores  = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        return torch.bmm(weights, V)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        attended = self.self_attention(outputs)
        outputs  = outputs + attended
        return outputs, hidden


class RNNDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, pretrained_embeddings):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(torch.tensor(pretrained_embeddings, dtype=torch.float32))
        self.embedding.weight.requires_grad = True
        self.attention_q = nn.Linear(hidden_dim, hidden_dim)
        self.attention_k = nn.Linear(hidden_dim, hidden_dim)
        self.attention_v = nn.Linear(hidden_dim, hidden_dim)
        self.rnn     = nn.RNN(input_size=embed_dim + hidden_dim, hidden_size=hidden_dim, batch_first=True)
        self.fc_out  = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.scale   = math.sqrt(hidden_dim)

    def attention(self, decoder_hidden, encoder_outputs):
        query = decoder_hidden.permute(1, 0, 2)
        Q = self.attention_q(query)
        K = self.attention_k(encoder_outputs)
        V = self.attention_v(encoder_outputs)
        scores  = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        weights = torch.softmax(scores, dim=-1)
        return torch.bmm(weights, V), weights

    def forward(self, tgt, hidden, encoder_outputs):
        tgt      = tgt.unsqueeze(1)
        embedded = self.dropout(self.embedding(tgt))
        context, weights = self.attention(hidden, encoder_outputs)
        rnn_input = torch.cat([embedded, context], dim=2)
        output, hidden = self.rnn(rnn_input, hidden)
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, weights


class RNNSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size     = src.shape[0]
        tgt_len        = tgt.shape[1]
        tgt_vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)
        encoder_outputs, hidden = self.encoder(src)
        decoder_input = tgt[:, 0]
        for t in range(1, tgt_len):
            prediction, hidden, _ = self.decoder(decoder_input, hidden, encoder_outputs)
            outputs[:, t, :] = prediction
            use_tf = torch.rand(1).item() < teacher_forcing_ratio
            decoder_input = tgt[:, t] if use_tf else prediction.argmax(1)
        return outputs

print('RNN model classes defined.')

In [ ]:
# ── Cell 6: LSTM Model Definition ────────────────────────────────────────────

class LSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, pretrained_embeddings):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(torch.tensor(pretrained_embeddings, dtype=torch.float32))
        self.embedding.weight.requires_grad = True
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)
        self.attention_q = nn.Linear(hidden_dim, hidden_dim)
        self.attention_k = nn.Linear(hidden_dim, hidden_dim)
        self.attention_v = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale   = math.sqrt(hidden_dim)

    def self_attention(self, hidden_states):
        Q = self.attention_q(hidden_states)
        K = self.attention_k(hidden_states)
        V = self.attention_v(hidden_states)
        scores  = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        return torch.bmm(weights, V)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)
        attended = self.self_attention(outputs)
        outputs  = outputs + attended
        return outputs, hidden, cell


class LSTMDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, pretrained_embeddings):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(torch.tensor(pretrained_embeddings, dtype=torch.float32))
        self.embedding.weight.requires_grad = True
        self.attention_q = nn.Linear(hidden_dim, hidden_dim)
        self.attention_k = nn.Linear(hidden_dim, hidden_dim)
        self.attention_v = nn.Linear(hidden_dim, hidden_dim)
        self.lstm    = nn.LSTM(input_size=embed_dim + hidden_dim, hidden_size=hidden_dim, batch_first=True)
        self.fc_out  = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.scale   = math.sqrt(hidden_dim)

    def attention(self, decoder_hidden, encoder_outputs):
        query = decoder_hidden.permute(1, 0, 2)
        Q = self.attention_q(query)
        K = self.attention_k(encoder_outputs)
        V = self.attention_v(encoder_outputs)
        scores  = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        weights = torch.softmax(scores, dim=-1)
        return torch.bmm(weights, V), weights

    def forward(self, tgt, hidden, cell, encoder_outputs):
        tgt      = tgt.unsqueeze(1)
        embedded = self.dropout(self.embedding(tgt))
        context, weights = self.attention(hidden, encoder_outputs)
        rnn_input = torch.cat([embedded, context], dim=2)
        output, (hidden, cell) = self.lstm(rnn_input, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell, weights


class LSTMSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size     = src.shape[0]
        tgt_len        = tgt.shape[1]
        tgt_vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        decoder_input = tgt[:, 0]
        for t in range(1, tgt_len):
            prediction, hidden, cell, _ = self.decoder(decoder_input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = prediction
            use_tf = torch.rand(1).item() < teacher_forcing_ratio
            decoder_input = tgt[:, t] if use_tf else prediction.argmax(1)
        return outputs

print('LSTM model classes defined.')

In [ ]:
# ── Cell 7: BiLSTM Model Definition ──────────────────────────────────────────
# BiLSTM uses HIDDEN_DIM=256 so bidirectional output = 256*2 = 512
BILSTM_HIDDEN = 256

class BiLSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, pretrained_embeddings):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding  = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(torch.tensor(pretrained_embeddings, dtype=torch.float32))
        self.embedding.weight.requires_grad = True
        self.bilstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim,
                               batch_first=True, bidirectional=True)
        self.fc_hidden   = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell     = nn.Linear(hidden_dim * 2, hidden_dim)
        self.norm_hidden = nn.LayerNorm(hidden_dim)
        self.norm_cell   = nn.LayerNorm(hidden_dim)
        self.attention_q = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.attention_k = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.attention_v = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.norm_enc    = nn.LayerNorm(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.scale   = math.sqrt(hidden_dim * 2)

    def self_attention(self, hidden_states):
        Q = self.attention_q(hidden_states)
        K = self.attention_k(hidden_states)
        V = self.attention_v(hidden_states)
        scores  = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        return torch.bmm(weights, V)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.bilstm(embedded)
        attended = self.self_attention(outputs)
        outputs  = self.norm_enc(outputs + attended)
        # Concat fwd + bwd hidden/cell, project to hidden_dim
        hidden = self.norm_hidden(torch.tanh(self.fc_hidden(
            torch.cat([hidden[-2], hidden[-1]], dim=1)
        ))).unsqueeze(0)
        cell = self.norm_cell(torch.tanh(self.fc_cell(
            torch.cat([cell[-2], cell[-1]], dim=1)
        ))).unsqueeze(0)
        return outputs, hidden, cell


class BiLSTMDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, pretrained_embeddings):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding  = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(torch.tensor(pretrained_embeddings, dtype=torch.float32))
        self.embedding.weight.requires_grad = True
        # Cross-attention: Q from decoder (hidden_dim), K/V from encoder (hidden_dim*2)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=4,
            dropout=dropout,
            batch_first=True,
            kdim=hidden_dim * 2,
            vdim=hidden_dim * 2
        )
        self.lstm    = nn.LSTM(input_size=embed_dim + hidden_dim, hidden_size=hidden_dim, batch_first=True)
        self.fc_out  = nn.Linear(hidden_dim * 2, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, hidden, cell, encoder_outputs):
        tgt      = tgt.unsqueeze(1)
        embedded = self.dropout(self.embedding(tgt))
        query    = hidden.permute(1, 0, 2)
        context, _ = self.cross_attention(query, encoder_outputs, encoder_outputs)
        rnn_input  = torch.cat([embedded, context], dim=2)
        output, (hidden, cell) = self.lstm(rnn_input, (hidden, cell))
        prediction = self.fc_out(torch.cat([output.squeeze(1), context.squeeze(1)], dim=1))
        return prediction, hidden, cell


class BiLSTMSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size     = src.shape[0]
        tgt_len        = tgt.shape[1]
        tgt_vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        decoder_input = tgt[:, 0]
        for t in range(1, tgt_len):
            prediction, hidden, cell = self.decoder(decoder_input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = prediction
            use_tf = torch.rand(1).item() < teacher_forcing_ratio
            decoder_input = tgt[:, t] if use_tf else prediction.argmax(1)
        return outputs

print('BiLSTM model classes defined.')

In [ ]:
# ── Cell 8: Shared Training + Evaluation Helpers ──────────────────────────────

def train_epoch(model, loader, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for src, tgt in tqdm(loader, desc='Train', leave=False):
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src, tgt, teacher_forcing_ratio=0.5)
        output = output[:, 1:, :].reshape(-1, output.shape[-1])
        tgt    = tgt[:, 1:].reshape(-1)
        loss   = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)


def evaluate_loss(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, tgt in tqdm(loader, desc='Val', leave=False):
            src, tgt = src.to(device), tgt.to(device)
            output   = model(src, tgt, teacher_forcing_ratio=0.0)
            output   = output[:, 1:, :].reshape(-1, output.shape[-1])
            tgt      = tgt[:, 1:].reshape(-1)
            epoch_loss += criterion(output, tgt).item()
    return epoch_loss / len(loader)


def train_model(model, model_name, n_epochs=N_EPOCHS):
    """Full training loop with checkpoint saving and auto-resume."""
    save_dir    = os.path.join(SAVE_DIR, model_name)
    os.makedirs(save_dir, exist_ok=True)
    latest_ckpt = os.path.join(save_dir, f'{model_name}_latest.pt')
    best_ckpt   = os.path.join(save_dir, f'{model_name}_best.pt')

    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    start_epoch   = 0
    best_val_loss = float('inf')

    if os.path.exists(latest_ckpt):
        ckpt = torch.load(latest_ckpt, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch   = ckpt['epoch'] + 1
        best_val_loss = ckpt['best_val_loss']
        print(f'[{model_name}] Resumed from epoch {ckpt["epoch"]} | best_val_loss={best_val_loss:.4f}')
    else:
        print(f'[{model_name}] Starting fresh training for {n_epochs} epochs...')

    for epoch in range(start_epoch, n_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP)
        val_loss   = evaluate_loss(model, val_loader, criterion)
        scheduler.step(val_loss)

        torch.save({
            'epoch': epoch, 'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_loss': best_val_loss
        }, latest_ckpt)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_ckpt)
            print(f'  Epoch {epoch+1:02d}/{n_epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f}  ✓ best saved')
        else:
            print(f'  Epoch {epoch+1:02d}/{n_epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')

    print(f'[{model_name}] Training complete! Best val loss: {best_val_loss:.4f}\n')
    return best_ckpt

print('Training helpers defined.')

In [ ]:
# ── Cell 9: Inference Helper ──────────────────────────────────────────────────

def indices_to_tokens(indices, idx2word):
    tokens = []
    for idx in indices:
        idx = idx.item() if hasattr(idx, 'item') else idx
        if idx == 2: break          # EOS
        if idx not in [0, 1]:       # skip PAD and SOS
            tokens.append(idx2word.get(idx, '<UNK>'))
    return tokens


def run_inference(model):
    """Run model on test set and return (hypotheses, references, sources)."""
    model.eval()
    hypotheses, references, sources = [], [], []
    for src, tgt in tqdm(test_loader, desc='Inference'):
        src, tgt = src.to(device), tgt.to(device)
        with torch.no_grad():
            output = model(src, tgt, teacher_forcing_ratio=0.0)
        pred_indices = output.argmax(dim=2)
        for i in range(pred_indices.shape[0]):
            hypotheses.append(' '.join(indices_to_tokens(pred_indices[i], idx2hi)))
            references.append(' '.join(indices_to_tokens(tgt[i],          idx2hi)))
            sources.append(   ' '.join(indices_to_tokens(src[i],          idx2eng)))
    return hypotheses, references, sources


def compute_metrics(hypotheses, references, sources, model_name):
    """Compute BLEU, ChrF++, BERTScore, COMET and return as dict."""
    from sacrebleu.metrics import BLEU, CHRF
    from bert_score import score as bert_score_fn

    print(f'  [{model_name}] Computing BLEU + ChrF++...')
    bleu_score = BLEU().corpus_score(hypotheses, [references])
    chrf_score = CHRF(word_order=2).corpus_score(hypotheses, [references])  # ChrF++

    print(f'  [{model_name}] Computing BERTScore...')
    _, _, F1_bert = bert_score_fn(
        hypotheses, references,
        lang='hi',
        model_type='bert-base-multilingual-cased',
        verbose=False
    )
    bert_f1 = F1_bert.mean().item()

    comet_score = None
    print(f'  [{model_name}] Computing COMET...')
    try:
        from comet import download_model, load_from_checkpoint
        comet_model_path = download_model('Unbabel/wmt22-comet-da')
        comet_model      = load_from_checkpoint(comet_model_path)
        comet_data   = [{'src': s, 'mt': h, 'ref': r}
                        for s, h, r in zip(sources, hypotheses, references)]
        comet_output = comet_model.predict(comet_data, batch_size=64, gpus=1)
        comet_score  = comet_output['system_score']
    except Exception as e:
        print(f'  [{model_name}] COMET skipped: {e}')

    return {
        'Model'     : model_name,
        'BLEU'      : round(bleu_score.score, 2),
        'ChrF++'    : round(chrf_score.score, 2),
        'BERTScore' : round(bert_f1, 4),
        'COMET'     : round(comet_score, 4) if comet_score is not None else 'N/A'
    }

print('Inference + metric helpers defined.')

In [ ]:
# ── Cell 10: Train RNN ────────────────────────────────────────────────────────
print('=' * 60)
print('  TRAINING MODEL 1/3: RNN + Self-Attention')
print('=' * 60)

rnn_encoder = RNNEncoder(len(eng_vocab), EMBED_DIM, HIDDEN_DIM, DROPOUT, eng_matrix).to(device)
rnn_decoder = RNNDecoder(len(hi_vocab),  EMBED_DIM, HIDDEN_DIM, DROPOUT, hi_matrix).to(device)
rnn_model   = RNNSeq2Seq(rnn_encoder, rnn_decoder, device).to(device)

params = sum(p.numel() for p in rnn_model.parameters() if p.requires_grad)
print(f'RNN trainable parameters: {params:,}')

rnn_best_ckpt = train_model(rnn_model, 'rnn')

# Free GPU memory before next model
del rnn_encoder, rnn_decoder
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# ── Cell 11: Train LSTM ───────────────────────────────────────────────────────
print('=' * 60)
print('  TRAINING MODEL 2/3: LSTM + Self-Attention')
print('=' * 60)

lstm_encoder = LSTMEncoder(len(eng_vocab), EMBED_DIM, HIDDEN_DIM, DROPOUT, eng_matrix).to(device)
lstm_decoder = LSTMDecoder(len(hi_vocab),  EMBED_DIM, HIDDEN_DIM, DROPOUT, hi_matrix).to(device)
lstm_model   = LSTMSeq2Seq(lstm_encoder, lstm_decoder, device).to(device)

params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f'LSTM trainable parameters: {params:,}')

lstm_best_ckpt = train_model(lstm_model, 'lstm')

del lstm_encoder, lstm_decoder
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# ── Cell 12: Train BiLSTM ─────────────────────────────────────────────────────
print('=' * 60)
print('  TRAINING MODEL 3/3: BiLSTM + Cross-Attention')
print('=' * 60)

bilstm_encoder = BiLSTMEncoder(len(eng_vocab), EMBED_DIM, BILSTM_HIDDEN, DROPOUT, eng_matrix).to(device)
bilstm_decoder = BiLSTMDecoder(len(hi_vocab),  EMBED_DIM, BILSTM_HIDDEN, DROPOUT, hi_matrix).to(device)
bilstm_model   = BiLSTMSeq2Seq(bilstm_encoder, bilstm_decoder, device).to(device)

params = sum(p.numel() for p in bilstm_model.parameters() if p.requires_grad)
print(f'BiLSTM trainable parameters: {params:,}')

bilstm_best_ckpt = train_model(bilstm_model, 'bilstm', n_epochs=15)  # BiLSTM trains for 15 epochs

del bilstm_encoder, bilstm_decoder
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# ── Cell 13: Evaluate All Models + Build Comparison Table ────────────────────
print('=' * 60)
print('  EVALUATING ALL MODELS ON TEST SET')
print('=' * 60)

all_results = []

# ── RNN evaluation ──
print('\n[1/3] Loading best RNN checkpoint...')
rnn_enc = RNNEncoder(len(eng_vocab), EMBED_DIM, HIDDEN_DIM, DROPOUT, eng_matrix).to(device)
rnn_dec = RNNDecoder(len(hi_vocab),  EMBED_DIM, HIDDEN_DIM, DROPOUT, hi_matrix).to(device)
rnn_eval = RNNSeq2Seq(rnn_enc, rnn_dec, device).to(device)
rnn_eval.load_state_dict(torch.load(rnn_best_ckpt, map_location=device))
hyp, ref, src = run_inference(rnn_eval)
all_results.append(compute_metrics(hyp, ref, src, 'RNN + Self-Attention'))
del rnn_enc, rnn_dec, rnn_eval; torch.cuda.empty_cache(); gc.collect()

# ── LSTM evaluation ──
print('\n[2/3] Loading best LSTM checkpoint...')
lstm_enc = LSTMEncoder(len(eng_vocab), EMBED_DIM, HIDDEN_DIM, DROPOUT, eng_matrix).to(device)
lstm_dec = LSTMDecoder(len(hi_vocab),  EMBED_DIM, HIDDEN_DIM, DROPOUT, hi_matrix).to(device)
lstm_eval = LSTMSeq2Seq(lstm_enc, lstm_dec, device).to(device)
lstm_eval.load_state_dict(torch.load(lstm_best_ckpt, map_location=device))
hyp, ref, src = run_inference(lstm_eval)
all_results.append(compute_metrics(hyp, ref, src, 'LSTM + Self-Attention'))
del lstm_enc, lstm_dec, lstm_eval; torch.cuda.empty_cache(); gc.collect()

# ── BiLSTM evaluation ──
print('\n[3/3] Loading best BiLSTM checkpoint...')
bi_enc = BiLSTMEncoder(len(eng_vocab), EMBED_DIM, BILSTM_HIDDEN, DROPOUT, eng_matrix).to(device)
bi_dec = BiLSTMDecoder(len(hi_vocab),  EMBED_DIM, BILSTM_HIDDEN, DROPOUT, hi_matrix).to(device)
bi_eval = BiLSTMSeq2Seq(bi_enc, bi_dec, device).to(device)
bi_eval.load_state_dict(torch.load(bilstm_best_ckpt, map_location=device))
hyp, ref, src = run_inference(bi_eval)
all_results.append(compute_metrics(hyp, ref, src, 'BiLSTM + Cross-Attention'))
del bi_enc, bi_dec, bi_eval; torch.cuda.empty_cache(); gc.collect()

print('\nAll evaluations complete!')

In [ ]:
# ── Cell 14: Final Comparison Table ──────────────────────────────────────────
import pandas as pd

results_df = pd.DataFrame(all_results)
results_df = results_df.set_index('Model')

print('\n')
print('=' * 65)
print('       FINAL MODEL COMPARISON — EN → HI TRANSLATION')
print('=' * 65)
print(results_df.to_string())
print('=' * 65)

# Highlight best per metric
print('\nBest per metric:')
for col in ['BLEU', 'ChrF++', 'BERTScore']:
    best_model = results_df[col].astype(float).idxmax()
    best_val   = results_df[col][best_model]
    print(f'  {col:<12}: {best_model}  ({best_val})')

# Save results to CSV
results_df.to_csv(os.path.join(SAVE_DIR, 'model_comparison.csv'))
print(f'\nResults saved to {SAVE_DIR}model_comparison.csv')

In [ ]:
# ── Cell 15: Sample Translations from All Models ──────────────────────────────
# Reload all 3 models for side-by-side sample comparison
N_SAMPLES = 5

# Get a small batch from test set
src_batch, tgt_batch = next(iter(test_loader))
src_batch = src_batch[:N_SAMPLES].to(device)
tgt_batch = tgt_batch[:N_SAMPLES].to(device)

def get_translations(model, src, tgt):
    model.eval()
    with torch.no_grad():
        output = model(src, tgt, teacher_forcing_ratio=0.0)
    pred = output.argmax(dim=2)
    return [' '.join(indices_to_tokens(pred[i], idx2hi)) for i in range(N_SAMPLES)]

# Load models
rnn_s    = RNNSeq2Seq(RNNEncoder(len(eng_vocab), EMBED_DIM, HIDDEN_DIM, DROPOUT, eng_matrix),
                       RNNDecoder(len(hi_vocab), EMBED_DIM, HIDDEN_DIM, DROPOUT, hi_matrix), device).to(device)
rnn_s.load_state_dict(torch.load(rnn_best_ckpt, map_location=device))

lstm_s   = LSTMSeq2Seq(LSTMEncoder(len(eng_vocab), EMBED_DIM, HIDDEN_DIM, DROPOUT, eng_matrix),
                        LSTMDecoder(len(hi_vocab), EMBED_DIM, HIDDEN_DIM, DROPOUT, hi_matrix), device).to(device)
lstm_s.load_state_dict(torch.load(lstm_best_ckpt, map_location=device))

bi_s     = BiLSTMSeq2Seq(BiLSTMEncoder(len(eng_vocab), EMBED_DIM, BILSTM_HIDDEN, DROPOUT, eng_matrix),
                          BiLSTMDecoder(len(hi_vocab), EMBED_DIM, BILSTM_HIDDEN, DROPOUT, hi_matrix), device).to(device)
bi_s.load_state_dict(torch.load(bilstm_best_ckpt, map_location=device))

rnn_trans    = get_translations(rnn_s,  src_batch, tgt_batch)
lstm_trans   = get_translations(lstm_s, src_batch, tgt_batch)
bilstm_trans = get_translations(bi_s,  src_batch, tgt_batch)

print('SAMPLE TRANSLATIONS')
print('=' * 75)
for i in range(N_SAMPLES):
    src_str = ' '.join(indices_to_tokens(src_batch[i].cpu(), idx2eng))
    ref_str = ' '.join(indices_to_tokens(tgt_batch[i].cpu(), idx2hi))
    print(f'Source     : {src_str}')
    print(f'Reference  : {ref_str}')
    print(f'RNN        : {rnn_trans[i]}')
    print(f'LSTM       : {lstm_trans[i]}')
    print(f'BiLSTM     : {bilstm_trans[i]}')
    print('-' * 75)